# Seminar Tag 05: Water vs. H₂ mit TCOCNNv3

Dieses Notebook liest vier CSV-Dateien ein, verwirft in **jeder Datei den ersten Zyklus**, trainiert einen binären TCOCNNv3-Klassifikator und prüft ihn anschließend auf getrennten Testdateien.

Erwartete Dateien: `train_water.csv`, `train_h2.csv`, `test_water.csv` und `test_h2.csv`. Jede Zeile ist ein vollständiger Zyklus mit 1.440 Werten. Bei sechs Zeilen bleiben nach dem Verwerfen des Einschwingzyklus fünf Beispiele pro Datei übrig.

> Wichtig: Fünf nutzbare Trainingszyklen pro Klasse reichen für eine Übung und einen technischen Funktionstest, aber nicht für eine belastbare Generalisierungsaussage. Für ein produktives Modell sollten mehrere unabhängige Messreihen/Boards aufgenommen werden.

## Warum Klassifikation statt künstlicher ppb-Ziele?

TCOCNNv3 kann direkt klassifizieren: `regression=False` aktiviert Cross-Entropy-Loss und `predict` liefert Klassenwahrscheinlichkeiten. Deshalb werden die echten Klassen `water` und `h2` benutzt. Die Ersatzwerte water = 400 ppb und H₂ = 20.000 ppb wären nur für eine Regressionsaufgabe sinnvoll und würden hier eine künstliche Ordnung und Distanz zwischen den Klassen einführen.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

# Repository-Wurzel unabhängig vom Startordner finden.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'Networks' / 'TCOCNNv3.py').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Repository-Wurzel mit Networks/TCOCNNv3.py nicht gefunden.')

DAY_DIR = ROOT / 'Evaluation Seminar' / 'Day_05'
DATA_DIR = DAY_DIR / 'data'  # Bei Bedarf auf den Ordner mit den vier CSVs ändern.

sys.path.insert(0, str(ROOT / 'Networks'))
sys.path.insert(0, str(DAY_DIR))

from TCOCNNv3 import TCOCNNv3Class
from day5_utils import (
    CLASS_NAMES, classification_metrics, log_zscore_per_sample,
    load_binary_split, set_seed, stratified_train_validation_split,
    train_classifier,
)

SEED = 42
set_seed(SEED)
print('Repository:', ROOT)
print('Datenordner:', DATA_DIR)
print('Gerät:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. CSV-Dateien prüfen und laden

Die Klasse wird aus dem Dateinamen abgeleitet. Dadurch sind keine zusätzlichen Target-Dateien nötig. Der Loader prüft außerdem Feldzahl, fehlende Werte und die Mindestanzahl an Zyklen.

In [ ]:
expected_files = [
    DATA_DIR / f'{split}_{class_name}.csv'
    for split in ('train', 'test')
    for class_name in CLASS_NAMES
]
missing = [path.name for path in expected_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f'Fehlende Dateien in {DATA_DIR}: {missing}. '
        'Lege die CSVs dort ab oder ändere DATA_DIR.'
    )

X_development, y_development, development_sources = load_binary_split(DATA_DIR, 'train')
X_test, y_test, test_sources = load_binary_split(DATA_DIR, 'test')

# Gewünschte Vorverarbeitung: erst log10, dann Z-Score je einzelnem Zyklus.
X_development_z = log_zscore_per_sample(X_development)
X_test_z = log_zscore_per_sample(X_test)

print('Entwicklung:', X_development.shape, 'Klassen:', np.bincount(y_development))
print('Test:', X_test.shape, 'Klassen:', np.bincount(y_test))
print('TCOCNN-Form: (Beispiele, parallele Sensoren, Zeitpunkte, Kanäle)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for class_index, class_name in enumerate(CLASS_NAMES):
    raw_axis = axes[0, class_index]
    prepared_axis = axes[1, class_index]
    for raw_cycle, prepared_cycle in zip(
        X_development[y_development == class_index, 0, :, 0],
        X_development_z[y_development == class_index, 0, :, 0],
    ):
        raw_axis.plot(raw_cycle, alpha=0.65)
        prepared_axis.plot(prepared_cycle, alpha=0.65)
    raw_axis.set(title=f'Rohsignal: {class_name}', xlabel='Zeitpunkt', ylabel='Sensorsignal')
    prepared_axis.set(
        title=f'log10 + Z-Score pro Zyklus: {class_name}',
        xlabel='Zeitpunkt',
        ylabel='Standardisiertes Signal',
    )
    raw_axis.grid(alpha=0.25)
    prepared_axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 2. Vorverarbeitete Zyklen in Train und Validierung trennen

Jeder Zyklus wurde bereits unabhängig verarbeitet: zuerst log10, danach Z-Score mit dem eigenen Mittelwert und der eigenen Standardabweichung. Deshalb werden keine gemeinsamen Skalierungswerte aus Train, Validierung oder Test gelernt. Die Batch-Normalisierung im TCOCNN übernimmt anschließend die Normalisierung der internen Aktivierungen.

In [ ]:
X_train, X_validation, y_train, y_validation = stratified_train_validation_split(
    X_development_z, y_development, validation_fraction=0.2, seed=SEED
)

print('Train:', X_train.shape, np.bincount(y_train))
print('Validierung:', X_validation.shape, np.bincount(y_validation))
print('Mittelwert je Train-Sample:', np.round(X_train.mean(axis=(1, 2, 3)), 4))
print('Std. je Train-Sample:', np.round(X_train.std(axis=(1, 2, 3)), 4))

## 3. TCOCNNv3 als Klassifikator trainieren

Wegen des sehr kleinen Datensatzes bleibt das Netz bewusst kompakt. Das Notebook speichert nach jeder Epoche den Zustand mit dem niedrigsten Validierungsverlust und stellt diesen Zustand am Ende wieder her.

In [ ]:
params = {
    'n_filter': 8,
    'section_depth': 3,
    'kernel': 15,
    'stride': 3,
    'convs_per_block': 2,
    'channel_growth': 8,
    'residual': True,
    'num_neurons': 32,
    'drop_out': 0.25,
}

model = TCOCNNv3Class(
    input_size=X_train.shape[1:],
    output_size=len(CLASS_NAMES),
    regression=False,
)
model.build_net(params)
model.compile_model(initial_learning_rate=1e-3)

history, best_epoch = train_classifier(
    model,
    X_train,
    y_train,
    X_validation,
    y_validation,
    epochs=40,
    batch_size=4,
)
print('Beste Epoche:', best_epoch)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validierung')
axes[0].set(title='Cross-Entropy-Loss', xlabel='Epoche', ylabel='Loss')
axes[1].plot(history['accuracy'], label='Train')
axes[1].plot(history['val_accuracy'], label='Validierung')
axes[1].set(title='Accuracy', xlabel='Epoche', ylabel='Anteil korrekt', ylim=(0, 1.05))
for axis in axes:
    axis.axvline(best_epoch - 1, color='black', linestyle='--', alpha=0.5)
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

## 4. Eingefrorenes Modell auf Validierung und Test prüfen

Der Test wird genau einmal nach der Modellauswahl ausgewertet. Zusätzlich zur Accuracy wird die Balanced Accuracy gezeigt; bei gleich großen Klassen sind beide ähnlich. Die Konfusionsmatrix macht sichtbar, welche Klasse verwechselt wurde.

In [ ]:
validation_probabilities = model.predict(X_validation)
test_probabilities = model.predict(X_test_z)

for name, truth, probabilities in (
    ('Validierung', y_validation, validation_probabilities),
    ('Test', y_test, test_probabilities),
):
    metrics = classification_metrics(truth, probabilities)
    print(f"{name}: Accuracy={metrics['accuracy']:.3f}, "
          f"Balanced Accuracy={metrics['balanced_accuracy']:.3f}")
    print(classification_report(
        truth, probabilities.argmax(axis=1), target_names=CLASS_NAMES,
        labels=[0, 1], zero_division=0
    ))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, title, truth, probabilities in (
    (axes[0], 'Validierung', y_validation, validation_probabilities),
    (axes[1], 'Test', y_test, test_probabilities),
):
    ConfusionMatrixDisplay.from_predictions(
        truth,
        probabilities.argmax(axis=1),
        labels=[0, 1],
        display_labels=CLASS_NAMES,
        cmap='Blues',
        colorbar=False,
        ax=axis,
    )
    axis.set_title(title)
fig.tight_layout()
plt.show()

In [ ]:
test_prediction = test_probabilities.argmax(axis=1)
print(f"{'Datei':<20} {'Zyklus':>6} {'Wahr':>8} {'Vorhersage':>12} {'P(water)':>10} {'P(h2)':>10}")
cycle_counter = {name: 1 for name in CLASS_NAMES}  # Originalzyklus 1 wurde verworfen.
for source, truth, prediction, probability in zip(
    test_sources, y_test, test_prediction, test_probabilities
):
    class_name = CLASS_NAMES[int(truth)]
    cycle_counter[class_name] += 1
    print(
        f'{source:<20} {cycle_counter[class_name]:>6} {class_name:>8} '
        f'{CLASS_NAMES[int(prediction)]:>12} {probability[0]:>10.3f} {probability[1]:>10.3f}'
    )

## 5. Modell und Vorverarbeitung speichern

Für eine spätere Vorhersage werden Gewichte, Architekturparameter, Klassenreihenfolge sowie die ausschließlich aus Trainingsdaten berechneten Skalierungswerte gemeinsam abgelegt.

In [ ]:
OUTPUT_DIR = DAY_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(
    {
        'state_dict': model.model.state_dict(),
        'input_size': tuple(X_train.shape[1:]),
        'class_names': CLASS_NAMES,
        'params': params,
        'regression': False,
    },
    OUTPUT_DIR / 'tcocnnv3_water_h2.pt',
)
np.savez(
    OUTPUT_DIR / 'tcocnnv3_water_h2_preprocessing.npz',
    method=np.asarray('log10_then_zscore_per_sample'),
    class_names=np.asarray(CLASS_NAMES),
)
print('Gespeichert in:', OUTPUT_DIR)

## Einordnung der Ergebnisse

- Eine sehr hohe Trainingsgenauigkeit bei schwacher Validierungs-/Testgenauigkeit ist bei nur wenigen Zyklen ein typisches Overfitting-Signal.
- Aufeinanderfolgende Zyklen derselben Aufnahme sind stark korreliert. Für eine realistische Bewertung sollten ganze Messreihen oder Boards getrennt werden, nicht einzelne Zyklen zufällig über Train und Test verteilt werden.
- Wenn später mehrere Dateien pro Klasse vorhanden sind, sollte die Validierung gruppiert nach Messreihe/Board erfolgen. Die vorgegebenen Testdateien bleiben weiterhin vollständig unangetastet.